# Camera Calibration with OpenCV

This notebook computes the intrinsic parameters and lens distortion coefficients of a camera from a set of chessboard images placed in the `data/` folder.

**Pipeline**
1. Load every image from `data/`.
2. Detect the inner corners of the chessboard pattern.
3. Refine corner positions to sub-pixel accuracy.
4. Run `cv.calibrateCamera` to estimate the camera matrix `K` and the distortion vector.
5. Evaluate the mean re-projection error and save the result.
6. Visualize undistortion on a sample image.

> **Chessboard convention.** `CHESSBOARD_SIZE = (cols, rows)` refers to the number of *inner* corners (the intersections between black and white squares), **not** the number of squares. A standard 8x7 squares board has `(7, 6)` inner corners.

## 1. Imports and parameters

In [ ]:
import glob
from pathlib import Path

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# --- Calibration parameters ---------------------------------------------------
DATA_DIR        = Path('data')        # folder with the chessboard pictures
IMAGE_GLOB      = '*.jpg'             # change to '*.png' / '*.*' if needed
CHESSBOARD_SIZE = (7, 6)              # inner corners (cols, rows)
SQUARE_SIZE_MM  = 25.0                # physical side of one square in millimeters
OUTPUT_FILE     = Path('calibration_result.npz')

# Sub-pixel refinement stop criteria
CRITERIA = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 1e-3)

## 2. Build the object-points template

Each chessboard corner is assigned a fixed 3D coordinate `(x, y, 0)` in the board reference frame. Multiplying by `SQUARE_SIZE_MM` yields the focal length in millimetres rather than pixel units of the square.

In [ ]:
cols, rows = CHESSBOARD_SIZE
objp = np.zeros((cols * rows, 3), np.float32)
objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
objp *= SQUARE_SIZE_MM

objpoints = []   # 3D points in the world reference frame
imgpoints = []   # 2D points in the image plane
used_files = []  # filenames where the pattern was successfully detected

## 3. Detect chessboard corners in every image

In [ ]:
image_paths = sorted(glob.glob(str(DATA_DIR / IMAGE_GLOB)))
assert image_paths, f'No images found in {DATA_DIR}/{IMAGE_GLOB}'
print(f'Found {len(image_paths)} candidate images.')

image_size = None  # (width, height) — taken from the first valid frame
preview = []       # store a few annotated frames for plotting later

for fname in image_paths:
    img = cv.imread(fname)
    if img is None:
        print(f'  [skip] could not read {fname}')
        continue
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    ret, corners = cv.findChessboardCorners(
        gray, CHESSBOARD_SIZE,
        flags=cv.CALIB_CB_ADAPTIVE_THRESH + cv.CALIB_CB_NORMALIZE_IMAGE,
    )

    if not ret:
        print(f'  [miss] {Path(fname).name}')
        continue

    corners2 = cv.cornerSubPix(gray, corners, (11, 11), (-1, -1), CRITERIA)
    objpoints.append(objp)
    imgpoints.append(corners2)
    used_files.append(fname)
    if image_size is None:
        image_size = gray.shape[::-1]  # (w, h)

    if len(preview) < 6:
        vis = img.copy()
        cv.drawChessboardCorners(vis, CHESSBOARD_SIZE, corners2, ret)
        preview.append((Path(fname).name, vis))

print(f'\nPattern detected in {len(used_files)} / {len(image_paths)} images.')
assert len(used_files) >= 5, 'Need at least 5 valid views for a reliable calibration.'

### Visualize the detected corners

In [ ]:
n = len(preview)
cols_plot = min(3, n)
rows_plot = int(np.ceil(n / cols_plot))
fig, axes = plt.subplots(rows_plot, cols_plot, figsize=(4 * cols_plot, 3 * rows_plot))
axes = np.atleast_1d(axes).ravel()
for ax, (name, vis) in zip(axes, preview):
    ax.imshow(cv.cvtColor(vis, cv.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=9)
    ax.axis('off')
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Solve for the intrinsics

`cv.calibrateCamera` returns:
- `K` — 3x3 camera matrix `[[fx, 0, cx], [0, fy, cy], [0, 0, 1]]`
- `dist` — distortion coefficients `(k1, k2, p1, p2, k3)` by default
- `rvecs`, `tvecs` — per-image extrinsics (rotation and translation of the board w.r.t. the camera)

In [ ]:
rms, K, dist, rvecs, tvecs = cv.calibrateCamera(
    objpoints, imgpoints, image_size, None, None,
)

print(f'RMS re-projection error (OpenCV):  {rms:.4f} px\n')
print('Camera matrix K =')
print(K)
print('\nDistortion coefficients (k1, k2, p1, p2, k3) =')
print(dist.ravel())

## 5. Mean re-projection error per image

The OpenCV RMS aggregates every corner. Per-image errors are useful to spot outlier views that should be removed and re-shot.

In [ ]:
per_image_err = []
total_pts = 0
total_err = 0.0
for i, (op, ip) in enumerate(zip(objpoints, imgpoints)):
    proj, _ = cv.projectPoints(op, rvecs[i], tvecs[i], K, dist)
    err = cv.norm(ip, proj, cv.NORM_L2) / len(proj)
    per_image_err.append(err)
    total_err += err ** 2 * len(proj)
    total_pts += len(proj)

mean_err = np.sqrt(total_err / total_pts)
print(f'Mean re-projection error: {mean_err:.4f} px\n')

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(per_image_err)), per_image_err)
ax.set_xlabel('image index')
ax.set_ylabel('mean error (px)')
ax.set_title('Per-image re-projection error')
plt.tight_layout()
plt.show()

## 6. Undistortion preview

`cv.getOptimalNewCameraMatrix` returns a refined matrix and a ROI that crops out the black borders introduced by the undistortion.

In [ ]:
sample = cv.imread(used_files[0])
h, w = sample.shape[:2]
new_K, roi = cv.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))
undistorted = cv.undistort(sample, K, dist, None, new_K)
x, y, rw, rh = roi
undistorted_cropped = undistorted[y:y + rh, x:x + rw]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv.cvtColor(sample, cv.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv.cvtColor(undistorted_cropped, cv.COLOR_BGR2RGB))
axes[1].set_title('Undistorted (cropped to ROI)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 7. Persist the calibration

Saving as `.npz` keeps the matrices ready for downstream scripts:

```python
data = np.load('calibration_result.npz')
K, dist = data['K'], data['dist']
```

In [ ]:
np.savez(
    OUTPUT_FILE,
    K=K,
    dist=dist,
    image_size=np.array(image_size),
    rms=np.array(rms),
    square_size_mm=np.array(SQUARE_SIZE_MM),
    chessboard_size=np.array(CHESSBOARD_SIZE),
)
print(f'Saved calibration to {OUTPUT_FILE.resolve()}')